# Lecture 6 — Model Answer
## Part-to-Whole: Hierarchical Visualization


In [ ]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


## Task 1 — Treemap: fossil fuel dependency by country


In [ ]:
# ── Data prep ─────────────────────────────────────────────────────────────────
df_fossil = df.loc[df['Source_Type'] == 'Fossil'].copy()

# CVD-safe palette for the three fossil sources
fossil_palette = {
    'Coal':        '#2E2E2E',   # dark charcoal: coal
    'Oil':         '#E07B39',   # orange: oil
    'Natural Gas': '#F5C842',   # yellow: natural gas
}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],  
    values='TWh',
    color='Source',                         
    color_discrete_map=fossil_palette,
    labels={'TWh': 'Energy (TWh)', 'Source': 'Fossil source'},
    title='Asia dominates fossil fuel consumption — coal drives China and India\'s outsized footprint',
    height=600, width=1100
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
fig.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',   # TWh values, no percentages
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='white',
)

# Grey out parent nodes (Region and Country level)
fig.data[0].marker.colors = [
    c if c in fossil_palette.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=55, b=10),
    paper_bgcolor='white',
)

fig.show()


## Task 2 — Sunburst: tipping behaviour by day and meal time


In [ ]:
# ── Data prep ─────────────────────────────────────────────────────────────────
tips = px.data.tips()

# Aggregate total bill per group — sum, not count
df_sun = (
    tips.groupby(['day', 'time', 'smoker'])['total_bill']
    .sum()
    .reset_index()
)

# CVD-safe blue/orange palette for smoker status
smoker_palette = {
    'No':  '#2E75B6',   # blue: non-smoker
    'Yes': '#E07B39',   # orange: smoker
}

# ── Step 1: Plotly Express base chart ─────────────────────────────────────────
fig = px.sunburst(
    df_sun,
    path=['day', 'time', 'smoker'],    
    values='total_bill',
    color='smoker',
    color_discrete_map=smoker_palette,
    labels={'total_bill': 'Total bill ($)', 'smoker': 'Smoker'},
    title='''Saturday dinner drives the most spending — non-smokers account for the majority across 3 out of 4 days''',
    width=900, height=600
)

# ── Step 2: Customisation ─────────────────────────────────────────────────────
fig.update_traces(
    textinfo='label+percent parent',   # % of parent segment
    hovertemplate='<b>%{label}</b><br>$%{value:,.0f}<br>%{percentParent:.0%} of %{parent}<extra></extra>',
    insidetextorientation='radial',    # text follows the curve of each segment
)

# Grey out parent nodes (day and time level)
fig.data[0].marker.colors = [
    c if c in smoker_palette.values() else '#DDDDDD'
    for c in fig.data[0].marker.colors
]

fig.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=55, b=10),
    paper_bgcolor='white',
)

fig.show()


## Task 3 — Treemap vs bar: low-carbon energy by country


In [ ]:
# ── Data prep ─────────────────────────────────────────────────────────────────
low_carbon = (
    df.loc[df['Source_Type'] == 'Low-carbon']
    .groupby('Country')['TWh']
    .sum()
    .reset_index()
    .sort_values('TWh')
)

# Dummy root node so Plotly has a named parent to anchor the treemap
low_carbon['All'] = 'Low-carbon'

# ── Treemap ───────────────────────────────────────────────────────────────────
fig_tree = px.treemap(
    low_carbon,
    path=['All', 'Country'],
    values='TWh',
    color='TWh',
    color_continuous_scale='Blues',
    title='Low-carbon energy by country (treemap)',
    labels={'TWh': 'Energy (TWh)'}, height=500
)
fig_tree.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>'
)
fig_tree.update_layout(
    font=dict(family='Arial', size=12),
    margin=dict(l=10, r=10, t=55, b=10),
    paper_bgcolor='white',
    coloraxis_showscale=False,
)
fig_tree.show()

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig_bar = px.bar(
    low_carbon,
    x='TWh',
    y='Country',
    orientation='h',
    color='TWh',
    color_continuous_scale='Blues',
    title='France, Norway and Canada lead on low-carbon energy',
    labels={'TWh': 'Low-carbon Energy (TWh)', 'Country': ''}, height=500
)
fig_bar.update_layout(
    plot_bgcolor='white',
    paper_bgcolor='white',
    font=dict(family='Arial', size=12),
    coloraxis_showscale=False,
    margin=dict(l=120, r=40, t=55, b=40),
)
fig_bar.update_xaxes(gridcolor='#EEEEEE')
fig_bar.update_yaxes(gridcolor='#EEEEEE')
fig_bar.show()


### Task 3 — reflection

**Which chart tells the ranking story more clearly, and why?**

The **bar chart** tells the ranking story more clearly. A shared baseline (x=0) means the eye can compare bar lengths directly and rank countries without effort. The treemap encodes value as rectangle area, which is harder to compare — humans are much better at judging length along a common axis than area. The treemap has one advantage: it makes the proportional dominance of the top countries visually striking at a glance. But if the question is *which country ranks where*, the bar chart answers it faster and more accurately.
